# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [16]:
# imports
import os
from openai import OpenAI
import ollama
from dotenv import load_dotenv
from IPython.display import display,Markdown,update_display

In [21]:
# constants

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2:3b'

In [3]:
# set up environment
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print("openai key exist")
else:
    print("openai key doesn't exist")

openai = OpenAI()

openai key exist


In [4]:
# here is the question; type over this to ask something new

question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

In [ ]:
# Get gpt-4o-mini to answer, with streaming
def chatGPT(question):
    system_propmt = "you are a helpfull assistant who answaers the given question."
    stream = openai.chat.completions.create(
        model=MODEL_GPT,
        messages=[
            {"role":"system","content":system_propmt},
            {"role":"user","content":question}
        ],
        stream=True
    )
    
    response = ""
    display_handle = display(Markdown(""), display_id=True)

    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        response = response.replace("```","").replace("markdown", "")
        update_display(Markdown(response), display_id=display_handle.display_id)


In [34]:
chatGPT(question)

<generator object chatGPT at 0x00000180E395CD00>

In [ ]:
# Get Llama 3.2 to answer
def lama(question):
    system_prompt = "You are a helpful assistant who answers the given question. Respond in markdown."
    
    stream = ollama.chat(
        model=MODEL_LLAMA,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question}
        ],
        stream=True
    )
    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    
    for chunk in stream:
        new_text = chunk["message"]["content"] or ''
        new_text = new_text.replace("```", "").replace("markdown", "")
        response += new_text
        update_display(Markdown(response), display_id=display_handle.display_id)
    


In [25]:
lama(question)

**Code Explanation**
===============

The given code uses Python's generator syntax to extract the authors of books from a list of dictionaries (`books`).

python
yield from {book.get("author") for book in books if book.get("author")}
```

Here's a breakdown:

* `yield from`: This keyword is used to delegate a sub-generator's iteration to another generator.
* `{...}`: This is a dictionary comprehension, which creates an iterator over the key-value pairs of the dictionary.
	+ `book.get("author")`: For each book in the list, tries to retrieve its "author" value from the dictionary using the `get()` method. If the "author" key does not exist, it defaults to `None`.
* `for book in books if book.get("author")`: This is a filter that only includes books with an existing "author" key.
* The `yield from` statement delegates this iterator to another generator (in this case, the dictionary comprehension).

**How it Works**
----------------

1. The code starts by creating a list of books (`books`) and assuming it contains dictionaries where each dictionary represents a book.
2. It then uses the dictionary comprehension to create an iterator over the authors of only those books that have an existing "author" key.
3. For each book in this filtered list, it tries to retrieve its author value using `book.get("author")`. If no such value exists, it defaults to `None`.
4. The resulting iterator (over the author values) is then delegated to another generator using `yield from`.
5. This means that instead of generating all the authors at once and storing them in a list or other data structure, this code generates them one by one, on demand.

**Why it's useful**
-----------------

This approach can be beneficial when:

* You need to process large datasets and don't want to load all the data into memory at once.
* You're working with streaming data sources (e.g., APIs, files) where data is generated on the fly.
* You want to reduce memory usage and only process the necessary data.

By using `yield from`, you can write more efficient code that processes data in a lazy manner, without having to store all the data in memory at once.

'**Code Explanation**\n===============\n\nThe given code uses Python\'s generator syntax to extract the authors of books from a list of dictionaries (`books`).\n\npython\nyield from {book.get("author") for book in books if book.get("author")}\n```\n\nHere\'s a breakdown:\n\n* `yield from`: This keyword is used to delegate a sub-generator\'s iteration to another generator.\n* `{...}`: This is a dictionary comprehension, which creates an iterator over the key-value pairs of the dictionary.\n\t+ `book.get("author")`: For each book in the list, tries to retrieve its "author" value from the dictionary using the `get()` method. If the "author" key does not exist, it defaults to `None`.\n* `for book in books if book.get("author")`: This is a filter that only includes books with an existing "author" key.\n* The `yield from` statement delegates this iterator to another generator (in this case, the dictionary comprehension).\n\n**How it Works**\n----------------\n\n1. The code starts by creating

In [35]:
def chatGPT_stream(question):
    system_propmt = "you are a helpfull assistant who answaers the given question."
    stream = openai.chat.completions.create(
        model=MODEL_GPT,
        messages=[
            {"role":"system","content":system_propmt},
            {"role":"user","content":question}
        ],
        stream=True
    )
    
    response = ""
    display_handle = display(Markdown(""), display_id=True)

    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response 


def lama_stream(question):
    system_prompt = "You are a helpful assistant who answers the given question. Respond in markdown."
    
    stream = ollama.chat(
        model=MODEL_LLAMA,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question}
        ],
        stream=True
    )
    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    
    for chunk in stream:
        new_text = chunk["message"]["content"] or ''
        yield new_text
    


In [36]:
import gradio as gr

def stream_model(question,model):
    if model == "gpt":
        result = chatGPT_stream(question)
    elif model == "lama":
        result = lama_stream(question)
    else:
        raise ValueError("Unknown model")
    yield from result


In [ ]:
gr.Interface(
    fn=stream_model,
    inputs=[gr.Textbox(label="Your message:"), gr.Dropdown(["gpt", "lama"], label="Select model", value="GPT")],
    outputs=[gr.Markdown(label="Response:")],
    flagging_mode="never"
).launch()

c:\Users\AERO\projects\llm_engineering\venv\Lib\site-packages\gradio\components\dropdown.py:230: UserWarning: The value passed into gr.Dropdown() is not in the list of choices. Please update the list of choices to include: GPT or set allow_custom_value=True.
  warnings.warn(


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
